In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from scipy import sparse

# 02 — Morgan fingerprints

**Input:** `processed/molecules_clean.csv` from `01_preprocessing.ipynb`

**Workflow in this notebook**

1. **Load** clean table (`Smiles`, `target`)
2. **Configure** fingerprint type and parameters (fixed for train + GA inference)
3. **Compute** Morgan bit vectors
4. **Check** featurization quality (failures, sparsity, duplicate fingerprints)
5. **Save** `X`, metadata, and config for `03_models`

## 1. Load clean data


In [2]:
DATA_DIR = Path('processed')
CLEAN_PATH = DATA_DIR / 'molecules_clean.csv'

df = pd.read_csv(CLEAN_PATH)
print(f'Rows: {len(df):,}')
print(f'Columns: {list(df.columns)}')
assert {'Smiles', 'target', 'Molecule ChEMBL ID'}.issubset(df.columns)
df[['Molecule ChEMBL ID', 'Smiles', 'target']].head(3)

Rows: 10,834
Columns: ['Molecule ChEMBL ID', 'Smiles', 'Molecule Name', 'Standard Value', 'pChEMBL Value', 'pIC50', 'pIC50_source', 'Standard Units', 'Target Name', 'target']


,Molecule ChEMBL ID,Smiles,target
0,CHEMBL10,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...,5.00
1,CHEMBL100714,COc1ccc(Nc2ncnc3cc(OC)c(OC)cc23)cc1OC,5.55
2,CHEMBL1009,N[C@@H](Cc1ccc(O)c(O)c1)C(=O)O,5.54


## 2. Fingerprint configuration

These values are saved to disk and must match at **training** and **GA inference** time.


In [3]:
FP_CONFIG = {
    'type': 'morgan',
    'radius': 2,
    'n_bits': 2048,
    'use_chirality': True,
    'use_bond_types': True,
}

FP_CONFIG

{'type': 'morgan',
 'radius': 2,
 'n_bits': 2048,
 'use_chirality': True,
 'use_bond_types': True}

## 3. Compute Morgan fingerprints


In [4]:
def build_morgan_generator(config):
    return rdFingerprintGenerator.GetMorganGenerator(
        radius=config['radius'],
        fpSize=config['n_bits'],
        includeChirality=config['use_chirality'],
        useBondTypes=config['use_bond_types'],
    )


def fp_to_row(bit_vect):
    arr = np.zeros((bit_vect.GetNumBits(),), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(bit_vect, arr)
    return arr


morgan_generator = build_morgan_generator(FP_CONFIG)

rows = []
failed_idx = []

for i, smiles in enumerate(df['Smiles']):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        failed_idx.append(i)
        continue
    rows.append(fp_to_row(morgan_generator.GetFingerprint(mol)))

if failed_idx:
    raise ValueError(f'RDKit failed on {len(failed_idx)} SMILES (expected 0 after preprocessing)')

X = sparse.csr_matrix(np.vstack(rows))
y = df['target'].to_numpy(dtype=np.float64)

print(f'X shape: {X.shape}  |  y shape: {y.shape}')
print(f'Sparsity: {1.0 - X.nnz / np.prod(X.shape):.4f}')

X shape: (10834, 2048)  |  y shape: (10834,)
Sparsity: 0.9698


## 4. Quality checks


In [5]:
bits_on = np.diff(X.indptr)
print('Bits set per molecule:')
print(pd.Series(bits_on).describe())

# Exact duplicate fingerprint vectors
fp_strings = [''.join(map(str, row)) for row in X.toarray()]
n_unique = len(set(fp_strings))
print(f'\nUnique fingerprint vectors: {n_unique:,} / {len(df):,}')
print(f'Duplicate vectors: {len(df) - n_unique:,}')

Bits set per molecule:
count    10834.000000
mean        61.921543
std         15.224185
min          2.000000
25%         51.000000
50%         63.000000
75%         73.000000
max        136.000000
dtype: float64

Unique fingerprint vectors: 10,674 / 10,834
Duplicate vectors: 160


## 5. Save artifacts

Row order in `X` matches `features_meta.csv` line-for-line (for `03_models`).


In [6]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

META_PATH = DATA_DIR / 'features_meta.csv'
X_PATH = DATA_DIR / 'X_morgan_r2_b2048.npz'
Y_PATH = DATA_DIR / 'y_pIC50.npy'
CONFIG_PATH = DATA_DIR / 'fingerprint_config.json'

meta = df[['Molecule ChEMBL ID', 'Smiles', 'target']].copy()
meta.to_csv(META_PATH, index=False)

sparse.save_npz(X_PATH, X)
np.save(Y_PATH, y)

with open(CONFIG_PATH, 'w') as f:
    json.dump(FP_CONFIG, f, indent=2)